# Simulador de Patrones de Radiación Acústica

Modelos implementados (formulación de cuadripolo acústico):
- **Monopolo:** fuente puntual, radiación omnidireccional → `(A/R) · e^{-jkR}`
- **Dipolo:** dos fuentes opuestas → `(A/R) · 2j·sin(k·d/2·sinθ)`
- **Tripolo ±:** combinación de tres fuentes → `(A/R) · [±1 ∓ 2cos(kd·sinθ)]`
- **Cuadripolo:** cuatro fuentes con coeficientes fraccionarios (7/8, 1/4, 3/4)

In [ ]:
import tkinter as tk
from tkinter import ttk
import numpy as np
import matplotlib.pyplot as plt

# ─── Configuración de parámetros físicos por defecto ──────────────────────────
CONFIG = {
    "frecuencia_hz":       100,   # Frecuencia de la fuente (Hz)
    "velocidad_sonido_ms": 343,   # Velocidad de propagación en aire (m/s)
    "separacion_m":        0.5,   # Separación entre fuentes del arreglo (m)
    "amplitud":            1.0,   # Factor de amplitud (adimensional)
    "distancia_m":         10.0,  # Distancia al punto de observación (m)
}
# ──────────────────────────────────────────────────────────────────────────────

def plot_graph():
    f = float(entry_frecuencia.get())
    c = float(entry_velocidad.get())
    d = float(entry_d.get())
    A = float(entry_A.get())
    R = float(entry_R.get())

    omega = 2 * np.pi * f          # Frecuencia angular (rad/s)
    k = omega / c                  # Número de onda (rad/m)
    Q = np.linspace(0, 2 * np.pi, 360)  # Ángulo de observación

    # Amplitudes complejas por tipo de fuente
    AMP_monopolo   = (A / R) * np.exp(-1j * k * R)
    AMP_dipolo     = (A / R) * (2j * np.sin(k * (d / 2) * np.sin(Q)))
    AMP_tripoloneg = (A / R) * (-1 + 2 * np.cos(k * d * np.sin(Q)))
    AMP_tripolopos = (A / R) * ( 1 - 2 * np.cos(k * d * np.sin(Q)))
    # Cuadripolo: superposición de cuatro fuentes con pesos fraccionarios
    AMP_cuadripolo = (A / R) * (
        (7/8) * np.cos((3/2) * k * d * np.sin(Q)) -
        (1/4) * np.cos(k * (d/2) * np.sin(Q)) +
        2j   * np.sin((3/2) * k * d * np.sin(Q)) -
        (3/4) * 1j * np.sin(k * (d/2) * np.sin(Q))
    )

    plt.figure(figsize=(12, 10))
    tipo_grafica = combo_tipo_grafica.get()

    AMP_MAP = {
        "Monopolo":         (AMP_monopolo,   np.ones_like(Q)),
        "Dipolo":           (AMP_dipolo,      None),
        "Tripolo positivo": (AMP_tripolopos,  None),
        "Tripolo negativo": (AMP_tripoloneg,  None),
        "Cuadripolo":       (AMP_cuadripolo,  None),
    }

    if tipo_grafica in AMP_MAP:
        amp, override = AMP_MAP[tipo_grafica]
        plt.subplot(1, 1, 1, polar=True)
        plt.title(tipo_grafica)
        r = np.abs(amp) * override if override is not None else np.abs(amp)
        plt.polar(Q, r)

    plt.tight_layout()
    plt.show()


root = tk.Tk()
root.title("Simulador de Patrones de Radiación")

fields = [
    ("Frecuencia (Hz):",                       "entry_frecuencia", CONFIG["frecuencia_hz"]),
    ("Velocidad de Propagación (m/s):",         "entry_velocidad",  CONFIG["velocidad_sonido_ms"]),
    ("Separación dipolo/tripolo/cuadripolo (m):","entry_d",         CONFIG["separacion_m"]),
    ("Amplitud:",                               "entry_A",          CONFIG["amplitud"]),
    ("Distancia al oyente (m):",                "entry_R",          CONFIG["distancia_m"]),
]

entries = {}
for row, (label_text, var_name, default) in enumerate(fields):
    ttk.Label(root, text=label_text).grid(column=0, row=row)
    e = ttk.Entry(root)
    e.insert(0, str(default))
    e.grid(column=1, row=row)
    entries[var_name] = e

entry_frecuencia = entries["entry_frecuencia"]
entry_velocidad  = entries["entry_velocidad"]
entry_d          = entries["entry_d"]
entry_A          = entries["entry_A"]
entry_R          = entries["entry_R"]

ttk.Label(root, text="Tipo de Gráfica:").grid(column=0, row=len(fields))
combo_tipo_grafica = ttk.Combobox(
    root,
    values=["Monopolo", "Dipolo", "Tripolo positivo", "Tripolo negativo", "Cuadripolo"],
    state='readonly'
)
combo_tipo_grafica.grid(column=1, row=len(fields))
combo_tipo_grafica.current(0)

ttk.Button(root, text="Graficar", command=plot_graph).grid(column=0, row=len(fields)+1, columnspan=2)

root.mainloop()
